In [8]:
import os
import sys
from ultralytics import settings, YOLO
from datetime import datetime
from clearml import Task, OutputModel
import torch
import boto3
from io import BytesIO
from PIL import Image
from torch.utils.data import Dataset


project_name="vision"
task_name="cat_dog"

# OBJECT_STORAGE_ENDPOINT = 'http://172.16.11.235:9000'  # Garage 서버 주소 (기본 포트 3900)
# AWS_ACCESS_KEY_ID = 'QQY84JF5HCFNC814TE75'
# AWS_ACCESS_SECRET_KEY = 'TOzA6qIU0smDLZhQFS7N8jRUVCB+RbdYoyhtJ3Ma'
# AWS_REGION = 'ap-northeast-2'

OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
AWS_REGION = 'ap-northeast-2'

access_key_id = AWS_ACCESS_KEY_ID
secret_access_key = AWS_ACCESS_SECRET_KEY
end_point = OBJECT_STORAGE_ENDPOINT
bucket_name = 'clearml-data'
region = AWS_REGION

is_dataset_created = False
is_previous_weight_created = False
ia_task_initiated = False


class MiniDatasetNotFound(Exception):
    def __init__(self, msg):
        self._msg = msg
    
    def __str__(self):
        return self._msg


class MinioDataset(Dataset):
    def __init__(self, endpoint_url, bucket_name, project_name, dataset_name, aws_access_key_id=None, aws_secret_access_key=None):
        # MinIO 연결 설정
        self._bucket_name = bucket_name
        self._project_name = project_name
        self._dataset_name = dataset_name
        self._aws_access_key_id = aws_access_key_id if aws_access_key_id is not None else AWS_ACCESS_KEY_ID
        self._aws_secret_access_key = aws_secret_access_key if aws_secret_access_key is not None else AWS_ACCESS_SECRET_KEY
        # 파일 목록만 미리 가져오기
        self._s3_client = boto3.client('s3',
            endpoint_url=endpoint_url,
            aws_access_key_id=self._aws_access_key_id,
            aws_secret_access_key=self._aws_secret_access_key
        )
        self._file_path = f'{self._project_name}/{self._dataset_name}'
        self._file_list = []

    def __len__(self):
        if len(self._file_list) == 0:
            self._file_list = [obj['Key'] for obj in self._s3_client.list_objects(Bucket=self._bucket_name, Prefix=self._file_path)['Contents']]
        return len(self._file_list)

    def getlist(self):
        if len(self._file_list) == 0:
            self._file_list = [obj['Key'] for obj in self._s3_client.list_objects(Bucket=self._bucket_name, Prefix=self._file_path)['Contents']]
        return self._file_list

    def __getitem__(self, idx):
        # 학습 시점에 해당 파일만 스트리밍으로 읽기
        if len(self._file_list) == 0:
            self._file_list = [obj['Key'] for obj in self._s3_client.list_objects(Bucket=self._bucket_name, Prefix=self._file_path)['Contents']]
        obj = self._s3_client.get_object(Bucket=self._bucket_name, Key=self._file_list[idx])
        img = Image.open(BytesIO(obj['Body'].read())).convert('RGB')
        return img

    def getitem(self, file_name: str):
        file_pathname = f'{self._file_path}/{file_name}'
        if len(self._file_list) == 0:
            self._file_list = [obj['Key'] for obj in self._s3_client.list_objects(Bucket=self._bucket_name, Prefix=self._file_path)['Contents']]
        if file_pathname in self._file_list:
            obj = self._s3_client.get_object(Bucket=self._bucket_name, Key=file_pathname)
            data = obj['Body'].read()
            return data 
        raise MiniDatasetNotFound(f'Can\'t found {file_name}')


# task = Task.init(
#     project_name="vision",
#     task_name="yollo8_test",
#     task_type=Task.TaskTypes.training
# )
ia_task_initiated = False

# task.set_base_docker(docker_image='python:3.12-bullseye')
# task.execute_remotely(queue_name='services')

"""
품질 -> brix(물질의 녿도),  -> 온도나 기타 환경 변화에 따라 brix 예측 모델 조사.
     특정 조건에 따라서, 작업시작/작업중단 등의 이벤트 알리기.
일정 시간에 적정 온도를 충족하지 못한 경우에 알림을 주는 코드 개발
산업 재해 데이터로 앞으로 있을 사고를 예게 (트리 모델로... )
"""

dataset = MinioDataset(OBJECT_STORAGE_ENDPOINT, bucket_name=bucket_name, project_name=project_name, dataset_name=task_name)
print(dataset.getlist())
## 데이터 경로 설정
data_filepath = 'coco8.yaml'
weight_filepath = 'yollov8n.pt'
try:
    data = dataset.getitem(data_filepath)
    print(data)
    with open(data_filepath, 'wb') as fd:
        fd.write(data)
    data = dataset.getitem(weight_filepath)
    with open(weight_filepath, 'wb') as fd:
        fd.write(data)
    # data_filepath = f'{os.getcwd()}{os.sep}data/yolo8/coco8.yaml'
    dataset_path = f'{os.getcwd()}{os.sep}datasets'
    weights_path = f'{os.getcwd()}{os.sep}weights'

    if os.path.isdir(dataset_path):
        print("Directory exists.")
    else:
        os.mkdir(dataset_path)
    is_dataset_created = True
            
    if os.path.isdir(weights_path):
        print("Directory exists.")
    else:
        os.mkdir(weights_path)
    is_previous_weight_created = True
    print(f"current datasets dir   = {settings['datasets_dir']}")
    settings.update({'datasets_dir': dataset_path})
    print(f"changed datasets dir   = {settings['datasets_dir']}")
    print(f"current weights  dir   = {settings['weights_dir']}")
    settings.update({'weights_dir': weights_path})
    print(f"changed weights  dir   = {settings['weights_dir']}")

    # 1. Load a pre-trained YOLOv8 nano model (fastest)
    model = YOLO(weight_filepath)
    
    # 2. Train the model on your custom dataset
    # data.yaml defines training/validation paths and classes
    results = model.train(
        data=data_filepath, 
        epochs=50, 
        imgsz=640, 
        batch=16,
        name='custom_yolov8_model'
    )

    # 3. 모델 검증 (Validation)
    metrics = model.val()
    print(f"Mean Average Precision (mAP): {metrics.box.map}")

    # 4. 이미지 추론 (Inference)
    # 로컬 이미지 경로, URL, 또는 폴더 경로 전달 가능
    results = model.predict(source='https://ultralytics.com/images/bus.jpg', save=True)
    # 5. 모델 내보내기 (Export)
    # ONNX, TensorRT, TFLite 등 다양한 포맷 지원
    weight_output_filename = model.export(format='onnx')
    output_model = OutputModel(task=task)
    # 로컬에 있는 파일을 서버로 업로드
    output_model.update_weights(
        weights_filename=weight_output_filename  # 업로드할 파일 경로
    )
    print(f"모델 업로드 완료: {weight_output_filename}")
except MiniDatasetNotFound as ex:
    print(f'Exception: {ex}')
finally:
    # -------------------------------
    # 8. Close task
    # -------------------------------
    # if ia_task_initiated:
    #     task.close()
    if is_dataset_created:
        os.remove(data_filepath)
    if is_previous_weight_created:
        os.remove(weight_filepath)

['vision/cat_dog/10057_jpg.rf.97fe48faf06eb4e0930d9ad78bbc15c9.jpg', 'vision/cat_dog/10057_jpg.rf.97fe48faf06eb4e0930d9ad78bbc15c9.txt', 'vision/cat_dog/10057_jpg.rf.9d3ebe61aec47998cd388f97b42d25b2.jpg', 'vision/cat_dog/10057_jpg.rf.9d3ebe61aec47998cd388f97b42d25b2.txt', 'vision/cat_dog/10057_jpg.rf.f4ac082571b349d3fd7521b54ffdd795.jpg', 'vision/cat_dog/10057_jpg.rf.f4ac082571b349d3fd7521b54ffdd795.txt', 'vision/cat_dog/10058_jpg.rf.6c79fed0a518f738444c1cffd6d56120.jpg', 'vision/cat_dog/10058_jpg.rf.6c79fed0a518f738444c1cffd6d56120.txt', 'vision/cat_dog/10058_jpg.rf.9c0b0244134288158167ef0e484672ab.jpg', 'vision/cat_dog/10058_jpg.rf.9c0b0244134288158167ef0e484672ab.txt', 'vision/cat_dog/10058_jpg.rf.9cf364e60d0fcffde0672593701ea54c.jpg', 'vision/cat_dog/10058_jpg.rf.9cf364e60d0fcffde0672593701ea54c.txt', 'vision/cat_dog/10059_jpg.rf.377ac80101629b9512a4ffbb2ed4c4eb.jpg', 'vision/cat_dog/10059_jpg.rf.377ac80101629b9512a4ffbb2ed4c4eb.txt', 'vision/cat_dog/10059_jpg.rf.696969b730805bedf5

In [ ]:
import os
import s3fs
from ultralytics import YOLO
from pathlib import Path

class MinioVFS:
    def __init__(self, endpoint, access_key, secret_key, bucket_name, project_name, dataset_name, classes: list = []):
        self.endpoint = endpoint
        self.bucket = bucket_name
        # s3fs 인터페이스 초기화
        self.fs = s3fs.S3FileSystem(
            key=access_key,
            secret=secret_key,
            client_kwargs={'endpoint_url': endpoint}
        )
        self.local_cache = Path("./minio_cache") / bucket_name
        self.local_cache.mkdir(parents=True, exist_ok=True)
        self._project_name = project_name
        self._dataset_name = dataset_name
        self._classes = classes

    def sync_metadata(self):
        """YAML 없이 클래스 정보를 자동으로 추출합니다."""
        # labels/train 폴더 내의 파일들을 분석하거나 전용 meta 파일을 읽음
        # 여기서는 폴더 구조 기반으로 클래스를 자동 매핑한다고 가정
        try:
            if len(self._classes) > 0:
                names = self._classes
            else:
                # MinIO 상의 경로: bucket/classes.txt 등을 읽어올 수 있음
                with self.fs.open(f"{self.bucket}/{self._project_name}/{self._dataset_name}/classes.txt", "rb") as f:
                    names = f.read().decode().splitlines()
        except:
            # 파일이 없으면 기본값 혹은 폴더명으로 대체
            names = ["class0", "class1"] 
        
        return names

    def smart_sync(self, remote_dir):
        """전체 데이터를 받지 않고, 변경된 파일만 로컬 캐시에 동기화 (최적화)"""
        remote_path = f"{self.bucket}/{remote_dir}"
        local_path = self.local_cache / remote_dir
        
        print(f"Syncing {remote_path} to {local_path}...")
        # s3fs의 get 기능을 활용해 변경된 내용만 효율적으로 다운로드
        self.fs.get(remote_path, str(local_path), recursive=True)

    def get_yolo_config(self):
        """YOLOv8 딕셔너리 설정을 자동 생성"""
        class_names = self.sync_metadata()
        
        # 학습 시작 전 필요한 데이터만 동기화 (images, labels)
        self.smart_sync(f"{self._project_name}/{self._dataset_name}")
        # self.smart_sync("val")

        return {
            "path": str(self.local_cache.absolute()), # 로컬 캐시 경로 전달
            "train": f"{self._project_name}/{self._dataset_name}",
            "val": f"{self._project_name}/{self._dataset_name}",
            "nc": len(class_names),
            "names": {i: name for i, name in enumerate(class_names)}
        }
    
    def get_yolo_config_path(self, config_filename="data.yaml"):
        """YOLOv8 딕셔너리 설정을 YAML 파일로 저장하고 경로 반환"""
        import yaml
        config = self.get_yolo_config()
        config_path = self.local_cache / config_filename
        with open(config_path, 'w') as f:
            yaml.dump(config, f)
        return str(config_path)


OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
AWS_REGION = 'ap-northeast-2'
PROJECT_NAME = "vision"
DATASET_NAME = "cat_dog"
BUCKET_NAME = "clearml-data"
CLASSES = ["cat", "dog"]

# --- 실제 사용 코드 ---
minio_vfs = MinioVFS(
    endpoint=OBJECT_STORAGE_ENDPOINT,
    access_key=AWS_ACCESS_KEY_ID,
    secret_key=AWS_ACCESS_SECRET_KEY,
    bucket_name=BUCKET_NAME,
    project_name=PROJECT_NAME,
    dataset_name=DATASET_NAME,
    classes=CLASSES
)

# 1. YAML 없이 설정값 생성 및 자동 동기화
import json
data_config = minio_vfs.get_yolo_config_path()
# json.dumps(data_config, indent=4)  # 설정값 확인용 출력

# 2. 모델 학습
model = YOLO("yolov8n.pt")
print(f"Training with config: {data_config}")
model.train(data=data_config, epochs=50, imgsz=640)


Syncing clearml-data/vision/cat_dog to minio_cache/clearml-data/vision/cat_dog...
Training with config: {'path': '/home/swhors/work/mlops_jupyter_notebooks-simpson/notebook/yolo8/minio_cache/clearml-data', 'train': 'vision/cat_dog', 'val': 'vision/cat_dog', 'nc': 2, 'names': {0: 'cat', 1: 'dog'}}
Ultralytics 8.4.48 🚀 Python-3.12.3 torch-2.11.0+cu130 CPU (Intel Core Ultra 5 225H)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data={'path': '/home/swhors/work/mlops_jupyter_notebooks-simpson/notebook/yolo8/minio_cache/clearml-data', 'train': 'vision/cat_dog', 'val': 'vision/cat_dog', 'nc': 2, 'names': {0: 'cat', 1: 'dog'}}, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'dict'

In [18]:
!pip install s3fs

In [17]:
!pip install boto3==1.43.0

  Attempting uninstall: boto3
    Found existing installation: boto3 1.43.6
    Uninstalling boto3-1.43.6:
      Successfully uninstalled boto3-1.43.6


In [15]:
!pip install --upgrade botocore

In [ ]:
import os
import subprocess
import time

OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
AWS_REGION = 'ap-northeast-2'
PROJECT_NAME = "vision"
DATASET_NAME = "cat_dog"
BUCKET_NAME = "clearml-data"
CLASSES = ["cat", "dog"]

def mount_minio_vfs(bucket_name, mount_path):
    # rclone 설정 (환경 변수 방식)
    os.environ["RCLONE_CONFIG_MYMINIO_TYPE"] = "s3"
    os.environ["RCLONE_CONFIG_MYMINIO_PROVIDER"] = "Minio"
    os.environ["RCLONE_CONFIG_MYMINIO_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
    os.environ["RCLONE_CONFIG_MYMINIO_SECRET_ACCESS_KEY"] = AWS_ACCESS_SECRET_KEY
    os.environ["RCLONE_CONFIG_MYMINIO_ENDPOINT"] = OBJECT_STORAGE_ENDPOINT

    os.makedirs(mount_path, exist_ok=True)

    # rclone 마운트 실행 (백그라운드)
    # --vfs-cache-mode minimal: 로컬에 복제하지 않고 실시간 읽기
    mount_cmd = [
        "rclone", "mount", f"myminio:{bucket_name}", mount_path,
        "--vfs-cache-mode", "minimal",
        "--vfs-read-chunk-size", "1M",
        # "--addr-use-vfs",
        "--daemon" # 백그라운드 실행
    ]
    
    subprocess.run(mount_cmd, check=True)
    
    # 마운트 완료 대기
    for _ in range(10):
        if os.path.ismount(mount_path):
            print(f"✅ MinIO mounted at {mount_path}")
            return
        time.sleep(1)
    raise Exception("❌ Mounting failed")

# ClearML 학습 코드 시작 부분에 추가
import os
mnt_path = "mnt/minio"
os.makedirs(mnt_path, exist_ok=True)
mount_minio_vfs(BUCKET_NAME, mnt_path)
dir_list = os.listdir(mnt_path)
for item in dir_list:
    print(item)
subprocess.run(["umount", mnt_path], check=True)
od.rmdir(mnt_path)

2026/05/10 23:30:26 NOTICE: Config file "/home/swhors/.config/rclone/rclone.conf" not found - using defaults


✅ MinIO mounted at mnt/minio
